# 05-04 Routing: 조건에 따라 다른 경로로 분기하기

웹 서비스에서 URL에 따라 다른 페이지를 보여주듯이, LLM 애플리케이션에서도 입력에 따라 다른 처리 체인을 선택해야 할 때가 있어요. 이것이 라우팅(Routing)이에요.

예를 들어, 고객 문의 챗봇을 만든다면 "결제 문의"는 결제 전문 체인으로, "기술 지원"은 기술 전문 체인으로 보내는 게 더 정확한 답변을 만들어줘요.

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있어요:

- 입력 데이터를 분류하는 체인을 구성하고, 분류 결과에 따라 다른 처리 경로로 라우팅(Routing)하는 패턴을 구현할 수 있어요
- `RunnableLambda` 방식과 `RunnableBranch` 방식의 차이를 설명하고, 상황에 맞게 선택할 수 있어요
- 분류 체인과 라우팅 체인을 파이프라인으로 연결하여 실제 챗봇 구조를 설계할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 다음 개념을 알고 있으면 좋아요:

- `RunnablePassthrough`와 LCEL 파이프 연산자(`|`) 사용법 (Ch05-01)
- `RunnableLambda`로 함수를 Runnable로 변환하는 방법 (Ch05-03)
- `operator.itemgetter`의 기본 사용법

---

> 🔑 **핵심 개념**: 라우팅은 "입력에 따라 다른 처리 경로를 선택하는 패턴"이에요. 하나의 체인이 모든 질문을 처리하는 대신, 질문 유형별로 전문화된 체인을 만들어 두고 적절한 체인으로 연결해요. 이렇게 하면 각 체인이 자기 분야에 집중할 수 있어 답변 품질이 올라가요.

아래 다이어그램은 이 노트북에서 구현할 조건 분기 흐름이에요:

```mermaid
flowchart TD
    Q["사용자 질문"]:::input
    C["분류 체인<br/>프로그래밍 / 요리 / 기타 판별"]:::process
    R["route() 함수<br/>분류 결과 확인"]:::process
    P["프로그래밍 전문가 체인"]:::process
    K["요리 전문가 체인"]:::process
    G["일반 체인"]:::process
    O["최종 답변"]:::output

    Q --> C --> R
    R -- "프로그래밍" --> P --> O
    R -- "요리" --> K --> O
    R -- "기타" --> G --> O

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

**라우팅을 구현하는 두 가지 방법:**

| 방법 | 핵심 도구 | 특징 |
|------|----------|------|
| **방법 1** (권장) | `RunnableLambda` | 일반 파이썬 함수로 분기 로직 작성 |
| **방법 2** | `RunnableBranch` | `(조건, 체인)` 튜플 리스트로 선언적 분기 |

두 방법 모두 동일한 기능을 제공하지만, LangChain 공식 문서에서는 첫 번째 방법을 권장해요.

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


True

In [2]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableBranch
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")


## 1. 질문 분류 체인 생성

라우팅의 첫 번째 단계는 **분류(Classification)**예요. 사용자 입력이 어떤 카테고리에 해당하는지 LLM이 먼저 판단해야, 그 결과를 기반으로 적절한 체인을 선택할 수 있어요.

이 예제에서는 질문을 `프로그래밍`, `요리`, 또는 `기타` 중 하나로 분류해요. 분류 결과가 이후 라우팅의 판단 기준이 되기 때문에, 정확하고 일관된 출력 형식이 중요해요.

> 🎯 **강의 포인트**: 분류 체인의 출력이 라우팅의 판단 기준이 돼요. LLM 출력은 비결정적(Non-deterministic)이기 때문에 "한 단어로만 응답하세요"처럼 출력 형식을 엄격하게 제약하는 것이 실무에서 중요해요. 출력 형식이 흔들리면 라우팅 로직이 예상대로 동작하지 않을 수 있어요.

> ⚠️ **주의**: 분류 프롬프트에 "한 단어로만 응답하세요" 조건을 포함하는 이유는 이후 `route()` 함수에서 문자열 포함 여부(`in` 연산자)로 분기를 결정하기 때문이에요. 예를 들어 LLM이 "이 질문은 프로그래밍에 관한 것입니다"라고 응답하면, `"프로그래밍" in topic` 조건은 참이 되지만 의도치 않은 부수 효과가 생길 수 있어요.

In [3]:
# ============================================================
# TODO: 입력 질문을 프로그래밍/요리/기타로 분류하는 체인을 완성하세요
# 힌트: ChatPromptTemplate.from_template(...) | model | StrOutputParser()
# 예상 결과: "파이썬 질문" → "프로그래밍", "김치찌개" → "요리", "날씨" → "기타"
# ============================================================

# ---------------------------------------------------
# 질문 분류 체인 생성 — 라우팅의 판단 기준
# ---------------------------------------------------

# 1단계: 분류 프롬프트 정의
# - "한 단어로만 응답"이 중요: 이후 route()에서 문자열 포함 여부로 분기하기 때문
classification_prompt = ChatPromptTemplate.from_template(
    """주어진 사용자 질문을 `프로그래밍`, `요리`, 또는 `기타` 중 하나로 분류하세요.
한 단어로만 응답하세요.

<question>
{question}
</question>

Classification:"""
)

# 2단계: 분류 체인 생성
# StrOutputParser(): AIMessage → str 로 변환 (문자열 비교에 필요)
classification_chain = (
    classification_prompt
    | model
    | StrOutputParser()
)

print("=" * 60)
print("✅ 분류 체인 생성 완료")
print("=" * 60)


✅ 분류 체인 생성 완료


In [4]:
# 분류 체인 테스트
test_questions = [
    "파이썬에서 리스트와 튜플의 차이는?",
    "김치찌개 만드는 방법은?",
    "오늘 날씨는 어때?"
]

print("=" * 60)
print("📊 질문 분류 테스트")
print("=" * 60)
for question in test_questions:
    category = classification_chain.invoke({"question": question})
    print(f"질문: {question}")
    print(f"분류: {category}")
    print()


📊 질문 분류 테스트
질문: 파이썬에서 리스트와 튜플의 차이는?
분류: 프로그래밍

질문: 김치찌개 만드는 방법은?
분류: 요리

질문: 오늘 날씨는 어때?
분류: 기타



## 2. 카테고리별 전문 체인 생성

분류 결과가 나오면, 각 카테고리에 맞는 전문 체인이 필요해요. 각 체인은 시스템 역할(System Prompt)을 다르게 설정하여 해당 분야에 특화된 답변을 제공해요.

> 💡 **실무 팁**: 실무에서는 각 카테고리별로 다른 모델을 사용하거나, 다른 RAG 소스를 연결하는 것도 가능해요. 예를 들어 프로그래밍 질문은 코드 검색 DB를, 요리 질문은 레시피 DB를 연결하는 식이에요. 여기서는 프롬프트만 다르게 설정하지만, 구조는 동일해요.

In [5]:
# ============================================================
# TODO: 각 카테고리에 맞는 전문 체인 3개(프로그래밍/요리/일반)를 정의하세요
# 힌트: ChatPromptTemplate.from_template(...) | model (파서 생략 가능)
# 예상 결과: 각 체인이 역할에 맞는 프롬프트로 응답
# ============================================================

# ---------------------------------------------------
# 카테고리별 전문 체인 — 역할 프롬프트(System Prompt) 분리
# ---------------------------------------------------

# 1단계: 프로그래밍 전문 체인
# "프로그래밍 전문가로서..."로 시작하도록 강제
programming_chain = (
    ChatPromptTemplate.from_template(
        """당신은 프로그래밍 전문가입니다.
항상 답변을 "프로그래밍 전문가로서..."로 시작하세요.

질문: {question}
답변:"""
    )
    | model
)

# 2단계: 요리 전문 체인
cooking_chain = (
    ChatPromptTemplate.from_template(
        """당신은 요리 전문가입니다.
항상 답변을 "요리 전문가로서..."로 시작하세요.

질문: {question}
답변:"""
    )
    | model
)

# 3단계: 일반 체인 (기타 카테고리용 기본 응답)
general_chain = (
    ChatPromptTemplate.from_template(
        """다음 질문에 간결하게 답변해주세요:

질문: {question}
답변:"""
    )
    | model
)

print("=" * 60)
print("✅ 카테고리별 체인 생성 완료")
print("=" * 60)


✅ 카테고리별 체인 생성 완료


## 3. RunnableLambda를 이용한 라우팅 (권장 방법)

LangChain 공식 문서에서 권장하는 방식이에요. 사용자 정의 함수를 `RunnableLambda`로 래핑하여 조건에 따라 적절한 체인을 **반환**해요.

이 방식의 핵심은 `route()` 함수가 **Runnable 객체를 반환**한다는 점이에요. 문자열이나 값을 반환하는 게 아니라, 실행 가능한 체인 자체를 반환하면 LCEL이 그 체인을 즉시 실행해요.

> 🔑 **핵심 개념**: `route()` 함수는 Runnable 객체를 반환해요. 반환된 Runnable이 즉시 실행되어 최종 결과를 만들어내요. "함수가 실행할 함수를 선택하는" 패턴이에요. 파이썬의 일급 객체(First-class Object) 개념과 연결되는 부분이에요.

> ⚠️ **주의**: `route()` 함수는 반드시 Runnable 객체를 반환해야 해요. 문자열을 반환하면 `TypeError`가 발생해요. 예: `return "프로그래밍"` (X) / `return programming_chain` (O)

> 🎯 **강의 포인트**: `RunnableLambda` 방식은 일반 파이썬 `if/elif/else` 문법을 그대로 쓸 수 있어서 가독성이 좋고, 복잡한 조건(정규식, 다중 조건 등)도 자유롭게 표현할 수 있어요. LangChain이 이 방식을 공식 권장하는 이유예요.

In [6]:
# ============================================================
# TODO: route() 함수를 작성하고 RunnableLambda로 감싸서 전체 체인을 구성하세요
# 힌트: def route(info): if "프로그래밍" in info["topic"]: return programming_chain ...
#       full_chain = {"topic": classification_chain, "question": itemgetter("question")} | RunnableLambda(route) | ...
# 예상 결과: 분류 결과에 따라 적절한 전문 체인으로 라우팅
# ============================================================

# ---------------------------------------------------
# RunnableLambda 라우팅 — 분류 결과로 체인 선택
# ---------------------------------------------------

# 1단계: 라우팅 함수 정의
# - 입력: {"topic": "프로그래밍", "question": "원본 질문"}  ← 이전 단계의 출력
# - 출력: Runnable 객체 (문자열이 아님!)
# - LCEL이 반환된 Runnable을 자동으로 실행해줌
def route(info: dict):
    """카테고리에 따라 적절한 체인을 반환하는 함수"""
    topic = info.get("topic", "").lower()
    
    # 분류 결과에 따라 전문 체인 선택
    if "프로그래밍" in topic:
        return programming_chain
    elif "요리" in topic:
        return cooking_chain
    else:
        return general_chain


# 2단계: 전체 체인 구성
# 딕셔너리 부분이 RunnableParallel로 자동 변환됨:
#   - "topic": classification_chain → 질문을 분류한 결과 문자열
#   - "question": itemgetter("question") → 원본 질문을 그대로 전달
# RunnableLambda(route): topic 값을 보고 실행할 체인을 동적으로 선택
# StrOutputParser(): AIMessage → str 변환
full_chain = (
    {
        "topic": classification_chain,
        "question": itemgetter("question")
    }
    | RunnableLambda(route)
    | StrOutputParser()
)

print("=" * 60)
print("✅ RunnableLambda 라우팅 체인 구성 완료")
print("=" * 60)

✅ RunnableLambda 라우팅 체인 구성 완료


In [7]:
# 프로그래밍 관련 질문 테스트
result1 = full_chain.invoke({"question": "파이썬에서 딕셔너리와 리스트의 차이는?"})

print("=" * 60)
print("📥 프로그래밍 질문")
print("=" * 60)
print(f"질문: 파이썬에서 딕셔너리와 리스트의 차이는?")
print()
print("📤 답변:")
print(result1)


📥 프로그래밍 질문
질문: 파이썬에서 딕셔너리와 리스트의 차이는?

📤 답변:
프로그래밍 전문가로서, 파이썬에서 딕셔너리와 리스트의 차이는 주로 데이터 구조와 접근 방식에 있습니다.

1. **데이터 구조**:
   - **리스트**: 순서가 있는 요소의 집합입니다. 리스트는 변경 가능(mutable)하며, 요소의 순서가 중요합니다. 요소는 인덱스를 통해 접근할 수 있습니다. 예를 들어, `my_list = [1, 2, 3]`와 같이 정의됩니다.
   - **딕셔너리**: 키-값 쌍의 집합으로, 순서가 없는 데이터 구조입니다. 딕셔너리도 변경 가능하며, 키를 사용하여 값에 접근합니다. 예를 들어, `my_dict = {'a': 1, 'b': 2}`와 같이 정의됩니다.

2. **접근 방식**:
   - **리스트**: 인덱스를 사용하여 요소에 접근합니다. 예를 들어, `my_list[0]`는 첫 번째 요소를 반환합니다.
   - **딕셔너리**: 키를 사용하여 값에 접근합니다. 예를 들어, `my_dict['a']`는 키 `'a'`에 해당하는 값을 반환합니다.

3. **용도**:
   - **리스트**는 순서가 중요한 데이터를 저장할 때 적합합니다. 예를 들어, 순차적으로 처리해야 하는 데이터를 다룰 때 유용합니다.
   - **딕셔너리**는 키를 사용해 데이터에 빠르게 접근해야 할 때 유용합니다. 즉, 특정 키에 대한 값을 빠르게 검색하고자 할 때 사용합니다.

이와 같이, 리스트와 딕셔너리는 서로 다른 특성과 용도가 있으므로 상황에 따라 적절한 자료구조를 선택하는 것이 중요합니다.


In [8]:
# 요리 관련 질문 테스트
result2 = full_chain.invoke({"question": "된장찌개 만드는 방법은?"})

print("=" * 60)
print("📥 요리 질문")
print("=" * 60)
print(f"질문: 된장찌개 만드는 방법은?")
print()
print("📤 답변:")
print(result2)


📥 요리 질문
질문: 된장찌개 만드는 방법은?

📤 답변:
요리 전문가로서... 된장찌개를 만드는 방법은 다음과 같습니다.

**재료:**
- 된장 2-3 큰술
- 물 4컵 (약 1리터)
- 두부 1/2 모 (사방 1cm로 잘라줍니다)
- 애호박 1/2개 (슬라이스)
- 대파 1대 (송송 썰기)
- 감자 1개 (큐브 모양으로 썰기)
- 양파 1/2개 (슬라이스)
- 고추 (청양고추 또는 빨간고추) 1-2개 (선택)
- 버섯 (표고버섯 또는 느타리버섯) 적당량 (슬라이스)
- 마늘 2-3 쪽 (다진 것)
- 참기름 약간
- 후춧가루 (선택)

**만드는 방법:**
1. 먼저, 냄비에 물을 넣고 가열합니다.
2. 물이 끓어오르면 된장을 체에 걸쳐 풀어줍니다. 된장을 직접 넣으면 덩어리가 남기 때문에 체에 걸러서 넣는 것이 좋습니다.
3. 된장 풀린 국물에 감자와 애호박을 먼저 넣고 5-7분 정도 끓입니다.
4. 감자가 어느 정도 익으면 양파, 두부, 버섯, 마늘을 넣고 다시 5분 정도 더 끓입니다.
5. 마지막으로 대파와 고추를 넣고 한소끔 더 끓인 후, 필요에 따라 후춧가루로 간을 맞춰줍니다.
6. 모든 재료가 잘 익고 국물이 맛있게 우러나면 불을 끄고, 그릇에 담아 참기름을 몇 방울 떨어뜨려 마무리합니다.

따뜻하게 제공하면 맛있는 된장찌개가 완성됩니다! 취향에 따라 해물이나 다른 채소를 추가해도 좋습니다. 즐거운 요리 되세요!


In [9]:
# 기타 질문 테스트
result3 = full_chain.invoke({"question": "인공지능의 미래는?"})

print("=" * 60)
print("📥 기타 질문")
print("=" * 60)
print(f"질문: 인공지능의 미래는?")
print()
print("📤 답변:")
print(result3)


📥 기타 질문
질문: 인공지능의 미래는?

📤 답변:
인공지능의 미래는 지속적인 발전과 혁신이 예상되며, 다양한 산업에서 자동화, 데이터 분석, 개인화된 서비스 제공 등으로 인간의 삶을 향상시킬 것입니다. 그러나 윤리, 일자리 변화, 데이터 보안 등의 문제도 함께 논의되어야 합니다.


## 4. RunnableBranch를 이용한 라우팅 (대안 방법)

`RunnableBranch`는 입력값에 따라 실행할 조건과 Runnable을 정의할 수 있는 특별한 유형의 `Runnable`이에요. `if/elif/else`와 비슷하지만, 선언적(Declarative)으로 작성하는 방식이에요.

**`RunnableBranch`의 동작 원리:**

1. `(조건_함수, 실행할_Runnable)` 튜플들의 리스트를 받아요
2. 호출 시 입력값을 각 조건 함수에 순서대로 전달해요
3. `True`를 반환하는 첫 번째 조건의 Runnable을 실행해요
4. 모든 조건이 `False`이면 마지막 인자(기본 Runnable)를 실행해요

> 💡 **실무 팁**: `RunnableLambda` 방식은 파이썬 표준 조건문으로 복잡한 로직(정규식, 다중 조건, 외부 API 호출 등)을 자유롭게 표현할 수 있어요. `RunnableBranch`는 단순한 조건 분기에 적합하지만, 조건이 복잡해질수록 `RunnableLambda` 방식이 유지보수하기 쉬워요.

> 🎯 **강의 포인트**: 두 방법을 나란히 비교하면 이해가 빨라요. `RunnableLambda`는 "파이썬 함수 안에서 if/else로 분기", `RunnableBranch`는 "조건-체인 쌍을 선언적으로 나열"하는 차이예요. 기능은 동일하지만 LangChain이 `RunnableLambda` 방식을 공식 권장한다는 점을 강조해보세요.

In [10]:
# ============================================================
# TODO: RunnableBranch를 사용하여 동일한 라우팅 로직을 구현하세요
# 힌트: RunnableBranch((조건_람다, 체인), ..., 기본_체인)
# 예상 결과: RunnableLambda 방식과 동일한 라우팅 결과
# ============================================================

# ---------------------------------------------------
# RunnableBranch: (조건, Runnable) 튜플 목록으로 분기 정의
# ---------------------------------------------------

# RunnableBranch 생성
# 구조: RunnableBranch( (조건1, 체인1), (조건2, 체인2), ..., 기본_체인 )
# - 각 조건은 lambda 함수로, 입력 딕셔너리를 받아 True/False 반환
# - 위에서 아래로 순서대로 평가하며, 첫 번째 True인 조건의 체인을 실행
# - 마지막 인자(튜플이 아닌 단독 Runnable)가 기본값(else) 역할
branch = RunnableBranch(
    # 조건 1: "프로그래밍"이 포함된 경우 → programming_chain 실행
    (lambda x: "프로그래밍" in x["topic"].lower(), programming_chain),
    # 조건 2: "요리"가 포함된 경우 → cooking_chain 실행
    (lambda x: "요리" in x["topic"].lower(), cooking_chain),
    # 기본값: 위 조건에 해당하지 않는 경우 → general_chain 실행
    general_chain,
)

# 전체 체인 구성 (RunnableLambda 방식과 동일한 입력 구조)
# 딕셔너리 → RunnableBranch → StrOutputParser
full_chain_branch = (
    {"topic": classification_chain, "question": itemgetter("question")}
    | branch
    | StrOutputParser()
)

print("=" * 60)
print("✅ RunnableBranch 라우팅 체인 구성 완료")
print("=" * 60)

✅ RunnableBranch 라우팅 체인 구성 완료


In [11]:
# RunnableBranch 체인 테스트
test_result = full_chain_branch.invoke({"question": "자바스크립트의 화살표 함수는?"})

print("=" * 60)
print("📥 RunnableBranch 테스트")
print("=" * 60)
print(f"질문: 자바스크립트의 화살표 함수는?")
print()
print("📤 답변:")
print(test_result)


📥 RunnableBranch 테스트
질문: 자바스크립트의 화살표 함수는?

📤 답변:
프로그래밍 전문가로서... 자바스크립트의 화살표 함수(Arrow Function)는 ES6(ECMAScript 2015)에서 도입된 간결한 함수 정의 방법입니다. 화살표 함수는 syntax가 더 짧고, 일반적인 함수 표현식보다 `this` 바인딩을 유지하는 방식이 다릅니다. 다음은 화살표 함수의 기본적인 사용법입니다.

```javascript
// 일반 함수 표현식
const add = function(a, b) {
    return a + b;
};

// 화살표 함수 표현식
const addArrow = (a, b) => a + b;

// 사용 예
console.log(add(2, 3));        // 출력: 5
console.log(addArrow(2, 3));   // 출력: 5
```

화살표 함수를 사용하는 주요 장점 중 하나는 `this` 키워드의 동작입니다. 화살표 함수 내에서는 주변 스코프의 `this` 값을 상속받기 때문에, 일반 함수처럼 새로운 `this` 컨텍스트를 생성하지 않습니다. 이 특징은 특히 콜백 함수나 클래스 메서드에서 유용하게 사용됩니다. 

예를 들어:

```javascript
class Counter {
    constructor() {
        this.count = 0;
    }

    increment() {
        setTimeout(() => {
            this.count++;  // 화살표 함수로 인해, 'this'는 Counter 인스턴스를 가리킴
            console.log(this.count);
        }, 1000);
    }
}

const counter = new Counter();
counter.increment();  // 1초 후 출력: 1
```

이렇게 화살표 함수는 코드의 간결함과 `this`의 예측 가능한 동작 덕분에 많은 개발자들에게 

## 정리

### 라우팅 방법 비교

| 항목 | `RunnableLambda` (권장) | `RunnableBranch` |
|------|------------------------|------------------|
| 코드 구조 | 일반 파이썬 함수 (`if/elif/else`) | 튜플 리스트 `(조건, 체인)` |
| 복잡한 조건 처리 | 용이 (정규식, 다중 조건, 외부 호출 등) | 제한적 (단순 조건에 적합) |
| 가독성 | 높음 (파이썬 개발자에게 익숙한 패턴) | 보통 (선언적 스타일에 익숙하면 좋음) |
| 디버깅 | 용이 (함수 내부에 print/로깅 추가 가능) | 제한적 |
| LangChain 권장 | **예** | 아니요 |

### 라우팅 패턴의 전체 흐름

```
입력 → 분류 체인 → route() 함수 → 전문 체인 선택 → 실행 → 최종 답변
```

### 핵심 요점

- 라우팅은 입력 분류 결과에 따라 서로 다른 체인으로 전달하는 패턴이에요
- 분류 체인의 출력 형식이 일관되어야 라우팅 로직이 안정적으로 동작해요
- `RunnableLambda`로 `route()` 함수를 래핑하면 일반 파이썬 조건문으로 분기 로직을 표현할 수 있어요
- `route()` 함수는 반드시 **Runnable 객체**를 반환해야 해요 (문자열이 아님)
- `RunnableBranch`는 기능적으로 동일하지만, LangChain은 `RunnableLambda` 방식을 권장해요

> 💡 **실무 팁**: 실무에서 라우팅은 언어별 처리(한국어/영어 질문 분기), 주제별 처리(결제/배송/반품 문의 분기), 복잡도별 처리(단순 질문은 작은 모델, 복잡한 질문은 큰 모델)에 활용해요. 분류의 정확도가 전체 시스템 품질을 좌우하므로, 분류 프롬프트 튜닝에 시간을 투자하는 것이 효과적이에요.

### 다음 노트북 예고

다음 노트북(`05-RunnableParallel.ipynb`)에서는 여러 체인을 동시에 실행하여 처리 속도를 높이는 `RunnableParallel` 패턴을 배워요. 순차 실행과 병렬 실행의 성능 차이를 직접 측정해볼게요.